**Microsoft Entra ID** (formerly Azure Active Directory) is Microsoft's cloud-based identity and access management service. It is the foundation of security in Azure — every user, application, and service that interacts with Azure must authenticate through Entra ID first.

If you are coming from AWS, the closest analogy is IAM — but Entra ID is significantly broader. It manages identities not just for Azure resources, but for Microsoft 365, thousands of third-party SaaS applications, and your own custom applications.

## Azure AD → Microsoft Entra ID

In **July 2023**, Microsoft renamed Azure Active Directory to **Microsoft Entra ID** as part of the broader Microsoft Entra product family.

| Old name | New name |
|---|---|
| Azure Active Directory (Azure AD) | Microsoft Entra ID |
| Azure AD P1 | Microsoft Entra ID P1 |
| Azure AD P2 | Microsoft Entra ID P2 |
| Azure AD B2C | Microsoft Entra External ID |
| Azure AD Domain Services | Microsoft Entra Domain Services |

The **functionality is identical** — only the branding changed. You will see both names used in documentation, the portal, and exam questions. This notebook uses **Entra ID** throughout, but treats both names as interchangeable.

## On-Premises Active Directory vs. Entra ID

Entra ID is **not** a cloud version of Windows Server Active Directory. They are different products built for different eras.

| | Windows Server AD | Microsoft Entra ID |
|---|---|---|
| **Protocol** | Kerberos, NTLM, LDAP | OAuth 2.0, OpenID Connect, SAML |
| **Structure** | Domains, forests, OUs | Flat — tenants, users, groups |
| **Queried via** | LDAP | REST API (Microsoft Graph) |
| **Designed for** | Corporate LAN | Internet-facing cloud apps |
| **Group Policy** | Yes | No (use Intune / Conditional Access instead) |
| **Managed by** | You (domain controllers) | Microsoft |

Many organisations run **both** — connecting them with **Microsoft Entra Connect** (formerly Azure AD Connect), which syncs on-premises identities to the cloud.

## Tenants

A **tenant** is a dedicated, isolated instance of Entra ID for your organisation. When a company signs up for Azure (or Microsoft 365), Microsoft automatically creates a tenant.

- Each tenant has a unique **Tenant ID** (a GUID) and a domain like `contoso.onmicrosoft.com`
- You can add custom domains: `contoso.com`
- Tenants are completely isolated — users in one tenant cannot see users in another
- One Azure subscription is associated with exactly one tenant, but one tenant can manage multiple subscriptions

```
Tenant: contoso.onmicrosoft.com
├── Subscription: Production
│   ├── Resource Group: webapp-rg
│   └── Resource Group: db-rg
├── Subscription: Development
│   └── Resource Group: dev-rg
└── Users, Groups, Applications
```

> The tenant is the **trust boundary** — all identity and access decisions happen within the context of a tenant.

## Users

A **user** in Entra ID represents a person who can sign in and access resources.

### User types

| Type | Description |
|---|---|
| **Member** | Belongs to your own tenant — employees, admins |
| **Guest** | External user invited from another tenant or personal account (B2B) |

### User identity sources

| Source | Description |
|---|---|
| **Cloud-only** | Created directly in Entra ID |
| **Synced from on-prem AD** | Via Entra Connect — managed on-premises, replicated to the cloud |
| **Federated** | Authenticated by on-premises AD FS — Entra ID trusts the on-prem identity provider |

### Global Administrator
The Global Administrator role in Entra ID has unrestricted access to all Entra ID settings. It is the equivalent of the AWS root account — enable MFA on it and avoid using it for daily work.

## Groups

**Groups** let you manage access for collections of users rather than individuals.

### Group types

| Type | Used for |
|---|---|
| **Security group** | Controlling access to Azure resources and applications |
| **Microsoft 365 group** | Collaboration — shared mailbox, calendar, Teams, SharePoint |

### Membership types

| Membership | How it works |
|---|---|
| **Assigned** | You manually add and remove users |
| **Dynamic user** | Membership rules based on user attributes — e.g. `department eq "Engineering"` |
| **Dynamic device** | Membership rules based on device attributes |

Dynamic groups are powerful: when a new employee is added with the right department attribute, they are automatically added to the correct group — and removed when they leave.

> Groups can be nested (a group can contain other groups) — unlike AWS IAM groups.

## Authentication Methods

Entra ID supports multiple ways for users to prove their identity.

### Multi-Factor Authentication (MFA)

MFA requires a second factor in addition to a password.

| Factor type | Examples |
|---|---|
| Something you know | Password, PIN |
| Something you have | Authenticator app, SMS code, hardware token, FIDO2 key |
| Something you are | Biometrics (Windows Hello) |

**Microsoft Authenticator** is the recommended app — it supports push notifications, TOTP codes, and passwordless phone sign-in.

### Passwordless Authentication

Entra ID supports signing in without a password at all:
- **Windows Hello for Business** — biometrics or PIN tied to a device
- **FIDO2 security keys** — physical hardware keys (YubiKey, etc.)
- **Microsoft Authenticator app** — approve a number match on your phone

### Self-Service Password Reset (SSPR)

Users can reset their own passwords without calling the help desk — using registered authentication methods like email, phone, or authenticator app. Requires Entra ID P1 or P2.

## App Registrations & Service Principals

When an **application** needs to access Azure resources or call Microsoft APIs, it must be registered in Entra ID.

### App Registration
An **App Registration** is the global definition of an application in your tenant — its identity, permissions, and credentials.

```
App Registration: my-web-app
├── Application (client) ID  →  unique GUID
├── Client secret or certificate  →  app's "password"
└── API permissions  →  what the app can access (e.g. Microsoft Graph)
```

### Service Principal
A **Service Principal** is the local instance of an App Registration within a specific tenant — think of it as the app's identity within your organisation.

| | App Registration | Service Principal |
|---|---|---|
| Scope | Global (home tenant) | Per tenant |
| Analogy | Class definition | Object instance |

### Authentication flows for apps

| Flow | Used when |
|---|---|
| Client credentials | App authenticates as itself (no user) — background services |
| Authorization code | App authenticates on behalf of a user — web apps |
| Device code | Apps on devices with no browser — CLI tools |

```python
from azure.identity import ClientSecretCredential

credential = ClientSecretCredential(
    tenant_id="<tenant-id>",
    client_id="<app-id>",
    client_secret="<secret>"
)
```

## Managed Identities

Managing client secrets and certificates for apps is error-prone — secrets expire, get leaked, or get hardcoded. **Managed Identities** solve this entirely.

A Managed Identity is an automatically managed service principal in Entra ID. Azure creates and rotates the credentials for you — **your code never handles a secret**.

### Two types

| Type | Lifecycle | Use case |
|---|---|---|
| **System-assigned** | Tied to the resource — deleted when the resource is deleted | One resource, one identity |
| **User-assigned** | Independent resource — survives the VM/function being deleted | Shared across multiple resources |

### How it works

```python
from azure.identity import DefaultAzureCredential
from azure.keyvault.secrets import SecretClient

# No secrets in code — uses Managed Identity automatically when running in Azure
credential = DefaultAzureCredential()
client = SecretClient(vault_url="https://my-vault.vault.azure.net", credential=credential)
secret = client.get_secret("db-password")
```

`DefaultAzureCredential` automatically uses Managed Identity when running inside Azure, and falls back to other methods (CLI login, environment variables) during local development.

### Supported resources
VMs, App Service, Azure Functions, AKS, Container Instances, Logic Apps, and many more.

> **Best practice:** Use Managed Identities for any Azure service that needs to access other Azure services. Never store credentials in code or config files.

## Conditional Access

**Conditional Access** is Entra ID's policy engine — it evaluates signals at sign-in time and decides whether to allow, block, or challenge the access request.

Think of it as: **If** [conditions are true] → **Then** [enforce controls]

### Signals (conditions)

| Signal | Example |
|---|---|
| **User / group** | Is this user a member of the Finance group? |
| **Location** | Is the sign-in coming from a trusted IP or known country? |
| **Device** | Is the device compliant and managed by Intune? |
| **Application** | Is the user accessing Salesforce or Azure portal? |
| **Sign-in risk** | Is Entra ID Protection flagging this as a risky sign-in? |
| **User risk** | Has this account been flagged as compromised? |

### Controls (actions)

| Control | Effect |
|---|---|
| **Block** | Deny access entirely |
| **Require MFA** | Challenge with a second factor |
| **Require compliant device** | Only allow access from Intune-managed devices |
| **Require password change** | Force a reset if the account is risky |
| **Limit session** | Restrict sign-in frequency or persistent browser sessions |

**Example policy:** Require MFA for all users signing into the Azure portal from outside the corporate network.

> Conditional Access requires **Entra ID P1** or P2.

## Entra ID Tiers

| Feature | Free | P1 | P2 |
|---|---|---|---|
| Users & groups | ✅ | ✅ | ✅ |
| SSO (limited apps) | ✅ | ✅ | ✅ |
| MFA (per-user) | ✅ | ✅ | ✅ |
| Conditional Access | ❌ | ✅ | ✅ |
| Self-Service Password Reset | ❌ | ✅ | ✅ |
| Dynamic groups | ❌ | ✅ | ✅ |
| Hybrid identity (Entra Connect) | ❌ | ✅ | ✅ |
| Identity Protection (risk-based) | ❌ | ❌ | ✅ |
| Privileged Identity Management | ❌ | ❌ | ✅ |
| Access Reviews | ❌ | ❌ | ✅ |

**P1** is included with Microsoft 365 Business Premium.  
**P2** adds risk-based intelligence and governance features — important for AZ-104 and security-focused roles.

## Privileged Identity Management (PIM)

**PIM** (Entra ID P2) implements **just-in-time privileged access** — users are eligible for admin roles but must explicitly activate them when needed, for a limited time window.

```
Without PIM:  User → always has Global Admin role
With PIM:     User → eligible for Global Admin
                    → activates role for 1 hour, with justification
                    → role expires automatically
```

Benefits:
- Reduces the attack surface — standing privileged access is a top risk
- Requires approval and justification for sensitive roles
- Full audit trail of when roles were activated and by whom
- Alerts when permanent role assignments are detected

> PIM is the Azure equivalent of AWS IAM's temporary credentials via STS — but with a governed workflow built on top.

## Working with Entra ID using Python

In [ ]:
# pip install azure-identity msgraph-sdk

In [ ]:
from azure.identity import DefaultAzureCredential
from msgraph import GraphServiceClient

# Authenticate using Managed Identity (in Azure) or CLI login (local dev)
credential = DefaultAzureCredential()
scopes = ['https://graph.microsoft.com/.default']

graph = GraphServiceClient(credential, scopes)

# List users in the tenant
users = await graph.users.get()
for user in users.value:
    print(f"{user.display_name:<30} {user.user_principal_name}")

In [ ]:
# List all security groups
groups = await graph.groups.get()
for group in groups.value:
    print(f"{group.display_name:<40} type: {group.group_types}")

In [ ]:
# List service principals (registered apps) in the tenant
sps = await graph.service_principals.get()
for sp in sps.value[:10]:  # first 10
    print(f"{sp.display_name:<40} appId: {sp.app_id}")

## Summary

| Concept | Key Takeaway |
|---|---|
| Entra ID | Microsoft's cloud identity provider — authentication for Azure, M365, and SaaS apps |
| Tenant | Isolated Entra ID instance for your organisation — the trust boundary |
| User | Member (internal) or Guest (B2B external) |
| Group | Security groups for access control; dynamic membership via attribute rules |
| MFA & Passwordless | Second factor required; FIDO2 and Authenticator app eliminate passwords |
| App Registration | Global app identity — client ID, secret/certificate, API permissions |
| Service Principal | Per-tenant instance of an App Registration |
| Managed Identity | Azure-managed credentials for services — no secrets in code |
| Conditional Access | Policy engine: if [signal] → then [control] — requires P1 |
| PIM | Just-in-time privileged access — activate roles on demand, time-limited — requires P2 |